In [1]:
import sys
sys.path.append('..')
from experiment_utils import read_config_file, get_site_splits
from utils import get_project_dir
from eval_utils import compare_aligned_data

import numpy as np
import os
import rasterio

import sklearn.metrics
import seaborn as sns
import matplotlib.pyplot as plt


import pandas as pd
import seaborn as sns

In [2]:
# to get the different test sets
config_fp = '../experiment_configs/train_baseline_local_models.yaml'
cfg_hp_search = read_config_file(config_fp)
cfg_data = cfg_hp_search['data']
split_seed = cfg_data['split_seed']


# 1. get the data for model type and number of layers

In [3]:
project_dir = '..' #get_project_dir()
outputs_dir = '/n/tambe_lab/Everyone/tree_mapping/model_output'

resolution= 10
reference_maps = ['ETH', 'GLAD']
models_rf = [f'local_only_models/rf/{x}' for x in ['rf_3_channel', 'rf_4_channel', 'rf_12_channel', 'rf_15_channel']] 
models_fcn = [f'local_only_models/128_filters/{x}' for x in ['3_channels', '4_channels', '12_channels', '15_channels']]
models_xception = [f'finetune_xceptionS2/pretrained/{x}' for x in ['1_layers_tuned', '2_layers_tuned']]

models = models_rf + models_fcn + models_xception

#pred padding is defined by how we cropped the satellite imagery to Karingani sites
pred_padding = 40

eval_metric = 'rmse'

lidar_coarsened_dir = f'{project_dir}/data/int/lidar/lidar_by_site_32736_{resolution}m'
sites = sorted([x for x in os.listdir(lidar_coarsened_dir) if not x.startswith('.')])

In [4]:
# Aggregate predictions and calculate performance metrics
results_by_site = {x: {} for x in reference_maps + models}
results_by_site_plot = []
results_aggregated = {x: {} for x in reference_maps + models}
results_by_split = {x: {} for x in reference_maps + models}

for model in models:
    labels = []
    preds = []
    model_output_dir = f'{outputs_dir}/{model}'
    for split_number in np.arange(4):
        labels_by_split = []
        preds_by_split = []
        test_sites = get_site_splits(split_seed, data_dir = '../data')[split_number]['test_sites']
        for site in test_sites:
            with rasterio.open(f'{lidar_coarsened_dir}/{site}/{site}_CHM_{resolution}m.tif') as file: # opens label tiff
                site_labels = file.read().ravel()
                labels = np.append(labels, site_labels)
                labels_by_split = np.append(labels_by_split, site_labels)

            with rasterio.open(f'{model_output_dir}/preds_{site}.tif') as file: # opens prediction tiff
                site_preds = file.read()[:,pred_padding:-pred_padding,pred_padding:-pred_padding].ravel()
                preds = np.append(preds, site_preds)
                preds_by_split = np.append(preds_by_split, site_preds)

            results_by_site[model][site] = compare_aligned_data(site_labels, site_preds)
    
        results_by_split[model][split_number] = compare_aligned_data(labels_by_split, preds_by_split)
    
    results_by_site_plot += [[results_by_site[model][site][eval_metric] for site in sites]]
    results_aggregated[model] = compare_aligned_data(labels, preds)
    
# get results
for reference_map in reference_maps:
    labels = []
    preds = []
    reference_map_by_site_dir = f'{project_dir}/data/existing_reference_data/{reference_map.lower()}_maps_per_site_{resolution}m'

    for split_number in np.arange(4):
        labels_by_split = []
        preds_by_split = []
        test_sites = get_site_splits(split_seed, data_dir = '../data')[split_number]['test_sites']
        for site in test_sites:
            with rasterio.open(f'{lidar_coarsened_dir}/{site}/{site}_CHM_{resolution}m.tif') as file: # opens label tiff
                site_labels = file.read().ravel()
                labels = np.append(labels, site_labels)
                labels_by_split = np.append(labels_by_split, site_labels)

            with rasterio.open(f'{reference_map_by_site_dir}/{reference_map}_MAP_{site}_{resolution}m.tif') as file: # opens prediction tiff
                site_preds = file.read().ravel()
                preds = np.append(preds, site_preds)
                preds_by_split = np.append(preds_by_split, site_preds)

            results_by_site[reference_map][site] = compare_aligned_data(site_labels, site_preds)
            
        results_by_split[reference_map][split_number] = compare_aligned_data(labels_by_split, preds_by_split)     
            
    results_by_site_plot += [[results_by_site[reference_map][site][eval_metric] for site in sites]]
    results_aggregated[reference_map] = compare_aligned_data(labels, preds)
    


In [5]:
# coalesce into pandas dataframes for plotting
results_keys = ['r2','mae','mse','rmse']
rows = []
for map_type, results_this_map_type in results_by_split.items():
    for split, results_this_split in results_this_map_type.items():
        results_this_type = {k: results_this_split[k] for k in results_keys}
        results_this_type.update({'map_type': map_type, 
                                  'test_split': split})
        rows.append(results_this_type)
results_by_split_df = pd.DataFrame(rows)


label_names_dict = {'ETH':'ETH', 
                    'GLAD':'GLAD', 
                    'local_only_models/128_filters/3_channels': 'FCN 3 channels (ours)',
                   'local_only_models/128_filters/4_channels': 'FCN 4 channels (ours)',
                   'local_only_models/128_filters/12_channels': 'FCN 12 channels (ours)',
                   'local_only_models/128_filters/15_channels': 'FCN 15 channels (ours)',
                   'local_only_models/rf/rf_3_channel': 'RF 3 channels (ours)',
                   'local_only_models/rf/rf_4_channel': 'RF 4 channels (ours)',
                    'local_only_models/rf/rf_12_channel': 'RF 12 channels (ours)',
                    'local_only_models/rf/rf_15_channel': 'RF 15 channels (ours)',
                    'finetune_xceptionS2/pretrained/1_layers_tuned': 'XceptionS2 1 layer tuned',
                    'finetune_xceptionS2/pretrained/2_layers_tuned': 'XceptionS2 2 layers tuned',
                   }

num_layers_dict = {'ETH': 'nan', 
                    'GLAD':'nan', 
                    'local_only_models/128_filters/3_channels': 3,
                   'local_only_models/128_filters/4_channels': 4,
                   'local_only_models/128_filters/12_channels': 12,
                   'local_only_models/128_filters/15_channels': 15,
                   'local_only_models/rf/rf_3_channel': 3,
                   'local_only_models/rf/rf_4_channel': 4,
                    'local_only_models/rf/rf_12_channel': 12,
                    'local_only_models/rf/rf_15_channel': 15,
                   'finetune_xceptionS2/pretrained/1_layers_tuned': 15,
                    'finetune_xceptionS2/pretrained/2_layers_tuned': 15,
                   }

results_by_split_df['model_name'] = results_by_split_df['map_type'].apply(lambda x: label_names_dict[x])
results_by_split_df['num_image_layers_used'] = results_by_split_df['map_type'].apply(lambda x: num_layers_dict[x])


In [6]:
def plot_panel(results_by_split_df, ax, y_key='rmse'):

    sns.scatterplot(data=results_by_split_df, x='model_name',y=y_key, 
                    s=100,
                    color='lightgrey', label='each of 4 test splits',
                   ax=ax)
    sns.scatterplot(data=results_by_split_df.groupby('model_name')[[y_key]].mean(), x='model_name', y=y_key, 
                    s=100,
                    color='forestgreen', label='average across test splits',
                   ax=ax)
    
    return ax

# 2. get the data for different number of training sites


In [ ]:
outputs_dir = '/n/tambe_lab/Everyone/tree_mapping/model_output/subsetted_training_sets'


df_rows = []

models_num_sites = ['rf_4_channel']
layers = 4
results_by_site = {x: {} for x in models_num_sites}
results_aggregated = {x: {} for x in models_num_sites}
results_by_split = {x: {} for x in models_num_sites}


for model in models_num_sites:
    for n_train_sites in [3,6,9,12]:
        for seed in range(10):
            labels = []
            preds = []
            model_output_dir = f'{outputs_dir}/{n_train_sites}_train_sites/seed_{seed}/{model}'
            for split_number in np.arange(4):
                labels_by_split = []
                preds_by_split = []
                test_sites = get_site_splits(split_seed, data_dir = '../data')[split_number]['test_sites']
                for site in test_sites:
                    with rasterio.open(f'{lidar_coarsened_dir}/{site}/{site}_CHM_{resolution}m.tif') as file: # opens label tiff
                        site_labels = file.read().ravel()
                        labels = np.append(labels, site_labels)
                        labels_by_split = np.append(labels_by_split, site_labels)

                    with rasterio.open(f'{model_output_dir}/preds_{site}.tif') as file: # opens prediction tiff
                        site_preds = file.read()[:,pred_padding:-pred_padding,pred_padding:-pred_padding].ravel()
                        preds = np.append(preds, site_preds)
                        preds_by_split = np.append(preds_by_split, site_preds)

                    results_by_site[model][site] = compare_aligned_data(site_labels, site_preds)
                res_dict = compare_aligned_data(labels_by_split, preds_by_split)
                results_by_split[model][split_number] = res_dict 
                res_dict.pop('errors') # dont need this for df
                this_row = {'model':model, 'seed': seed, 'num_train_sites': n_train_sites, 'split': split_number, 'layers':layers}
                this_row.update(res_dict)
                
                df_rows.append(this_row)

            results_by_site_plot += [[results_by_site[model][site][eval_metric] for site in sites]]
            results_aggregated[model] = compare_aligned_data(labels, preds)

In [ ]:
results_by_num_splits = pd.DataFrame(df_rows)

In [ ]:
#save this as a CSV for loading later
results_by_num_splits.to_csv('../experiment_results/results_by_num_training_sites.csv')

In [ ]:
results_with_subsetted_sites = pd.read_csv('../experiment_results/results_by_num_training_sites.csv')
results_with_subsetted_sites = results_with_subsetted_sites.rename(columns={'split':'test_split', 'layers':'num_image_layers_used'})
results_with_subsetted_sites = results_with_subsetted_sites.loc[:,results_keys + ['seed','test_split', 'num_image_layers_used', 'num_train_sites']]
# #results_with_subsetted_sites.loc[:,'model_name'] = 'FCN 4 channels (ours)'

groups = ['test_split','num_train_sites']
grouped = results_with_subsetted_sites.groupby(groups)

results_by_num_sites = pd.DataFrame()
rows = []
for name, group in grouped:
    row_df = pd.DataFrame(group.mean()).T
    row_df.loc[:,'model_name'] = f'RF ({name[1]} train sites)'
    row_df.loc[:,groups[0]] = name[0]
    row_df.loc[:,groups[1]] = name[1]
    
    results_by_num_sites= pd.concat((results_by_num_sites,row_df))
    
results_by_num_sites.sort_values(by='num_train_sites', inplace=True)

In [ ]:
y_key = 'rmse'
metric_labels = {'mae': "Mean Absolute Error (m)", 'rmse': "Root Mean Squared Error (m)"}

fig, ax = plt.subplots(1,3, figsize=(12,4), sharey=True)

results_by_model_type = results_by_split_df[results_by_split_df['model_name'].apply(lambda x: x in ['FCN 4 channels (ours)',  'XceptionS2 2 layers tuned', 'RF 4 channels (ours)'])]
plot_panel(results_by_model_type, ax[0], y_key='rmse');

methods_by_sentinel_layers = ['FCN 3 channels (ours)', 'FCN 4 channels (ours)', 'FCN 12 channels (ours)', 'FCN 15 channels (ours)']
results_by_sentinel_layers = results_by_split_df[results_by_split_df['model_name'].apply(lambda x: x in methods_by_sentinel_layers)]
plot_panel(results_by_sentinel_layers, ax[1], y_key='rmse');
plot_panel(results_by_num_sites, ax[2], y_key='rmse');

for i in range(len(ax)):
    ax[i].set_xlabel(None)

for ax_this in ax:
    ax_this.set_xticklabels(ax_this.get_xticklabels(), 
                          rotation=45, horizontalalignment='right')
    ax_this.legend([])
    
ax[0].set_ylabel(metric_labels[y_key])
ax[0].set_title('model architecture')
ax[1].set_title('number of Sentinel-2 layers')
ax[2].set_title('number of training sites')


sns.despine()